## **Hybrid RAG hallucination verification**

In [1]:
!pip install torch transformers datasets rank_bm25 sentence-transformers pymorphy3 bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 111.8 MB/s eta 0:00:00


In [2]:
import json
import os
import re
import numpy as np
import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import pymorphy3
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device.upper()}")
if device == "cpu":
    print("ВНИМАНИЕ: Смените тип среды выполнения на GPU (T4).")

# 1. Dataset SberQuAD
print("\nЗагрузка валидационной выборки SberQuAD...")
raw_dataset = load_dataset("kuznetsoffandrey/sberquad", split="validation")

# 2. Extracting contexts
print("Создание базы знаний...")
documents = []
for item in raw_dataset:
    context = item.get('context', '').strip()
    if context:
        documents.append(context)

documents = list(set(documents))
print(f"База знаний успешно создана! Загружено {len(documents)} уникальных чанков.")

# 3. Morphology and Tokenizer for BM25
print("Инициализация pymorphy3 и лемматизация документов для BM25...")
morph = pymorphy3.MorphAnalyzer()

def russian_lemmatized_tokenizer(text):
    words = re.findall(r'\b[а-яА-Яa-zA-Z0-9_]+\b', text.lower())
    return [morph.parse(word)[0].normal_form for word in words]

tokenized_documents = [russian_lemmatized_tokenizer(doc) for doc in documents]
bm25 = BM25Okapi(tokenized_documents)

# 4. Embedding
print("Загрузка мультиязычной модели эмбеддингов LaBSE...")
embedder = SentenceTransformer("sentence-transformers/LaBSE", device=device)
doc_embeddings = embedder.encode(documents, convert_to_tensor=True, normalize_embeddings=True)

Используемое устройство: CUDA

Загрузка валидационной выборки SberQuAD...


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

sberquad/train-00000-of-00001.parquet:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

sberquad/validation-00000-of-00001.parqu(…):   0%|          | 0.00/3.43M [00:00<?, ?B/s]

sberquad/test-00000-of-00001.parquet:   0%|          | 0.00/4.93M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

In [3]:
# 5. Loading LLM (Using Qwen 2.5)
print("\nЗагрузка квантованных моделей (4-bit NF4)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Qwen2.5-7B is well-suited for Russian-language RAG
model_id = "Qwen/Qwen2.5-7B-Instruct"
print(f"Загрузка единой базовой модели: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)

# Generation function
def ask_model(messages, max_tokens=150):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return generated_text.strip()

# Hybrid search function
def hybrid_search(query, top_k=3):
    tokenized_query = russian_lemmatized_tokenizer(query)
    bm25_scores = bm25.get_scores(tokenized_query)

    query_embedding = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    cos_scores = util.cos_sim(query_embedding, doc_embeddings)[0].cpu().numpy()

    # Avoid division by zero with empty scores
    bm25_max, bm25_min = np.max(bm25_scores), np.min(bm25_scores)
    bm25_norm = (bm25_scores - bm25_min) / (bm25_max - bm25_min + 1e-9) if (bm25_max - bm25_min) > 0 else bm25_scores

    cos_max, cos_min = np.max(cos_scores), np.min(cos_scores)
    cos_norm = (cos_scores - cos_min) / (cos_max - cos_min + 1e-9) if (cos_max - cos_min) > 0 else cos_scores

    hybrid_scores = 0.5 * bm25_norm + 0.5 * cos_norm
    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]
    return [documents[idx] for idx in top_indices]


Загрузка квантованных моделей (4-bit NF4)...
Загрузка единой базовой модели: Qwen/Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [7]:
# 6. RAG Pipeline
# ==========================================
def run_rag_agent(question):
    print(f"\n[Агент]: Запускаю гибридный поиск по запросу: '{question}'...")
    retrieved_chunks = hybrid_search(question, top_k=3)
    context = "\n\n---\n\n".join(retrieved_chunks)

    # Шаг 1: Генерация ответа
    gen_messages = [
        {
            "role": "system",
            "content": (
                "Вы — точный русскоязычный ассистент. Ответьте на вопрос пользователя, строго основываясь на предоставленном Контексте.\n"
                "КРИТИЧЕСКИЕ ПРАВИЛА:\n"
                "1. Отвечайте максимально кратко (1-2 предложения), формулируйте только прямой факт.\n"
                "2. Пишите ответ СТРОГО на русском языке. Используйте ТОЛЬКО кириллицу для всех имён и названий (например, 'Имхотеп', а не 'Имhotep').\n"
                "3. Запрещено додумывать логику, вводить причинно-следственные связи или детали, которых нет в тексте.\n"
                "4. Если в тексте нет прямого ответа, напишите строго одну фразу: 'Я не знаю'."
            )
        },
        {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"}
    ]
    initial_answer = ask_model(gen_messages, max_tokens=80)
    print(f"[Первичный ответ]: {initial_answer}\n")

    # Шаг 2: Верификация
    eval_messages = [
        {
            "role": "system",
            "content": (
                "Вы — строгий эксперт по верификации ответов в RAG-системах.\n"
                "Ваша задача — проверить, подтверждается ли 'Ответ' предоставленным 'Контекстом'.\n"
                "ПРАВИЛА ВАЛИДАЦИИ:\n"
                "1. Если ответ обрывается на полуслове (например, 'содерж'), содержит латинские буквы там, где нужна кириллица, или явно додуман — это ГАЛЛЮЦИНАЦИЯ.\n"
                "2. Если ключевой факт и имена переданы верно, не придирайтесь к порядку слов или синонимам.\n"
                "3. Напишите 1-2 предложения краткого анализа, а на последней строке выведите строго одно слово: ДА (если ответ корректен) или НЕТ (если это галлюцинация/брак)."
            )
        },
        {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}\nОтвет модели: {initial_answer}\n\nВыведите обоснование и вердикт (ДА/НЕТ) в самом конце:"}
    ]
    verdict_raw = ask_model(eval_messages, max_tokens=100).upper()
    print(f"[Рассуждение и вердикт валидатора]:\n{verdict_raw}")

    print("\n=== ИТОГОВЫЙ РЕЗУЛЬТАТ ===")

    last_line = verdict_raw.split('\n')[-1].strip()

    if "ДА" in last_line or verdict_raw.endswith("ДА"):
        print("Перекрестная проверка пройдена успешно. Ответ верифицирован.")
        return initial_answer
    else:
        print("ОБНАРУЖЕНА ГАЛЛЮЦИНАЦИЯ! Сгенерированный ответ заблокирован.")
        return "Ошибка: Ответ заблокирован модулем безопасности."

# Тестовый запуск на проблемном месте
test_question = "Когда египетский врач Имхотеп впервые описал некоторые органы и их функции?"
result = run_rag_agent(test_question)


[Агент]: Запускаю гибридный поиск по запросу: 'Когда египетский врач Имхотеп впервые описал некоторые органы и их функции?'...
[Первичный ответ]: XXVII век до н. э.

[Рассуждение и вердикт валидатора]:
В КОНТЕКСТЕ УКАЗАНО, ЧТО ЕГИПЕТСКИЙ ВРАЧ ИМХОТЕП ОПИСАЛ НЕКОТОРЫЕ ОРГАНЫ И ИХ ФУНКЦИИ В XXVII ВЕКЕ ДО Н. Э. ОТВЕТ МОДЕЛИ СОВПАДАЕТ С ЭТОЙ ИНФОРМАЦИЕЙ, НЕ СОДЕРЖИТ ОШИБОК И НЕ ТРЕБУЕТ ДОПОЛНИТЕЛЬНЫХ УТОЧНЕНИЙ.

ДА

=== ИТОГОВЫЙ РЕЗУЛЬТАТ ===
Перекрестная проверка пройдена успешно. Ответ верифицирован.


## **RAG Evaluation**

In [8]:
import re
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# 1. Defining the External Judge
# =====================================================================
def ask_external_judge_fixed(context, question, initial_answer):
    judge_messages = [
        {
            "role": "system",
            "content": (
                "Вы — независимый эксперт-аудитор. Ваша задача — проверить, корректен ли 'Ответ модели'.\n"
                "Инструкция по оценке:\n"
                "Ответ считается НЕКОРРЕКТНЫМ (БРАК/ГАЛЛЮЦИНАЦИЯ), только если он:\n"
                "- Содержит китайские иероглифы или явный обрыв строки на полуслове.\n"
                "- Написан с использованием смеси русских и английских букв в ОДНОМ слове.\n"
                "- Напрямую противоречит контексту или содержит ложные факты, искажающие суть текста.\n\n"
                "ОБРАТИТЕ ВНИМАНИЕ:\n"
                "- Написание иностранных имен кириллицей (например, 'Имхотеп') является КОРРЕКТНЫМ.\n"
                "- Ответ 'Я не знаю' при отсутствии данных в тексте является КОРРЕКТНЫМ.\n\n"
                "Формат ответа: На первой строке напишите строго одно из двух слов: КОРРЕКТЕН или ГАЛЛЮЦИНАЦИЯ. Затем, со следующей строки, приведите краткое пояснение."
            ),
        },
        {
            "role": "user",
            "content": f"Контекст:\n{context}\n\nВопрос:\n{question}\n\nОтвет модели:\n{initial_answer}",
        },
    ]

    result = ask_model(judge_messages, max_tokens=200).strip().upper()
    first_line = result.split('\n')[0]

    if "КОРРЕКТЕН" in first_line and "НЕ" not in first_line and "ГАЛЛЮЦИНАЦИЯ" not in first_line:
        return 1
    else:
        return 0


# 2. Main data collection and evaluation pipeline
# =====================================================================
def evaluate_optimized_rag_to_excel(test_samples=50):
    print(f"\n[Тест] Начинаем сбор данных для оптимизированного RAG ({test_samples} примеров)...")

    records = []


    sampled_data = raw_dataset.select(range(min(test_samples, len(raw_dataset))))

    for idx, item in enumerate(sampled_data):
        question = item.get("question", "")
        print(f"Обработка: Тест {idx+1}/{test_samples}...", end="\r")

        # 1. Context Retrieval
        retrieved_chunks = hybrid_search(question, top_k=3)
        context = "\n\n---\n\n".join(retrieved_chunks)

        # 2. Initial Answer
        gen_messages = [
            {
                "role": "system",
                "content": (
                    "Вы — точный русскоязычный ассистент. Ответьте на вопрос пользователя, строго основываясь на предоставленном Контексте.\n"
                    "КРИТИЧЕСКИЕ ПРАВИЛА:\n"
                    "1. Отвечайте максимально кратко (1-2 предложения), формулируйте только прямой факт.\n"
                    "2. Пишите ответ СТРОГО на русском языке. Используйте ТОЛЬКО кириллицу для всех имён и названий (например, 'Имхотеп', а не 'Имhotep').\n"
                    "3. Запрещено додумывать логику, вводить причинно-следственные связи или детали, которых нет в тексте.\n"
                    "4. Если в тексте нет прямого ответа, напишите строго одну фразу: 'Я не знаю'."
                )
            },
            {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"}
        ]
        initial_answer = ask_model(gen_messages, max_tokens=80)

        # 3. Verification (Internal Validator/Critic)
        eval_messages = [
            {
                "role": "system",
                "content": (
                    "Вы — строгий эксперт по верификации ответов в RAG-системах.\n"
                    "Ваша задача — проверить, подтверждается ли 'Ответ' предоставленным 'Контекстом'.\n"
                    "ПРАВИЛА ВАЛИДАЦИИ:\n"
                    "1. Если ответ обрывается на полуслове (например, 'содерж'), содержит латинские буквы там, где нужна кириллица, или явно додуман — это ГАЛЛЮЦИНАЦИЯ.\n"
                    "2. Если ключевой факт и имена переданы верно, не придирайтесь к порядку слов или мелким синонимам.\n"
                    "3. Напишите 1-2 предложения краткого анализа, а на последней строке выведите строго одно слово: ДА (если ответ корректен) или НЕТ (если это галлюцинация/брак)."
                )
            },
            {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}\nОтвет модели: {initial_answer}\n\nВыведите обоснование и вердикт (ДА/НЕТ) в самом конце:"}
        ]
        verdict_raw = ask_model(eval_messages, max_tokens=100).upper()


        verdict_lines = [line.strip() for line in verdict_raw.split('\n') if line.strip()]
        last_line_critic = verdict_lines[-1] if verdict_lines else ""

        critic_label = 1 if ("ДА" in last_line_critic or verdict_raw.endswith("ДА")) else 0

        # 4. External Judge Execution
        judge_label = ask_external_judge_fixed(context, question, initial_answer)

        # Сбор лога для Excel
        records.append({
            "id": idx + 1,
            "question": question,
            "context": context,
            "initial_answer": initial_answer,
            "critic_reasoning": verdict_raw,  # Обоснование Критика
            "critic_verdict": critic_label,   # Бинарный ответ Критика (1/0)
            "judge_verdict": judge_label,     # Бинарный ответ Судьи (1/0)
            "human_label": ""                 # Оставляем пустым для ручной разметки
        })

    # Compiling logs for the Excel spreadsheet
    df = pd.DataFrame(records)
    df.to_excel("optimized_rag_marking.xlsx", index=False)
    print(f"\n[УСПЕХ] Обработано {len(records)} примеров. Файл 'optimized_rag_marking.xlsx' успешно сохранен и готов к скачиванию!")

# Saving dataframe to an Excel file
evaluate_optimized_rag_to_excel(test_samples=50)


[Тест] Начинаем сбор данных для оптимизированного RAG (50 примеров)...

[УСПЕХ] Обработано 50 примеров. Файл 'optimized_rag_marking.xlsx' успешно сохранен и готов к скачиванию!


In [9]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def calculate_rag_metrics(file_path="optimized_rag_marking.xlsx"):
    df = pd.read_excel(file_path)

    # Check for empty rows in human labels
    if df['human_label'].isnull().any() or (df['human_label'] == "").any():
        print("[Предупреждение]: Вы заполнили не все строки в human_label. Пустые строки будут пропущены.")
        df = df.dropna(subset=['human_label'])

    y_true = df['human_label'].astype(int).tolist()
    y_critic = df['critic_verdict'].astype(int).tolist()
    y_judge = df['judge_verdict'].astype(int).tolist()

    if not y_true:
        print("[ОШИБКА]: Колонка 'human_label' пустая! Пожалуйста, проставь 1 или 0 в Excel и загрузите файл обратно.")
        return

    # Helper function to print a clean report
    def print_report(name, y_pred):
        print("\n" + "="*60)
        print(f" МЕТРИКИ ДЛЯ КОМПОНЕНТА: {name}")
        print("="*60)
        print(f"Accuracy (Доля правильных решений): {accuracy_score(y_true, y_pred):.2f}")
        print("\nДетальная статистика классификации:")
        print(classification_report(y_true, y_pred, target_names=["Брак/Галл (0)", "Корректен (1)"], zero_division=0))

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        print("-"*60)
        print(f"True Negative  (Успешно заблокировано брака) : {tn}")
        print(f"False Positive (ПРОПУЩЕНА ГАЛЛЮЦИНАЦИЯ!)     : {fp}")
        print(f"False Negative (Ошибочный блок хорошего)     : {fn}")
        print(f"True Positive  (Успешно одобрено хорошего)   : {tp}")
        print("="*60)

    # Run calculations separately for both systems
    print_report("ВНУТРЕННИЙ ВАЛИДАТОР", y_critic)
    #print_report("ВНЕШНИЙ ЭКСПЕРТ-АУДИТОР (СУДЬЯ)", y_judge)

# Calculate final safety metrics for the RAG agent
calculate_rag_metrics("optimized_rag_marking.xlsx")


 МЕТРИКИ ДЛЯ КОМПОНЕНТА: ВНУТРЕННИЙ ВАЛИДАТОР
Accuracy (Доля правильных решений): 0.90

Детальная статистика классификации:
               precision    recall  f1-score   support

Брак/Галл (0)       0.62      0.71      0.67         7
Корректен (1)       0.95      0.93      0.94        43

     accuracy                           0.90        50
    macro avg       0.79      0.82      0.80        50
 weighted avg       0.91      0.90      0.90        50

------------------------------------------------------------
True Negative  (Успешно заблокировано брака) : 5
False Positive (ПРОПУЩЕНА ГАЛЛЮЦИНАЦИЯ!)     : 2
False Negative (Ошибочный блок хорошего)     : 3
True Positive  (Успешно одобрено хорошего)   : 40
